# 1 Import, Config, Const



In [1]:
!pip install openai

from google.colab import userdata
import openai
import json
import time
import pandas as pd
import os

In [ ]:
# models for groq
models_G = ['openai/gpt-oss-20b',
            'llama-3.1-8b-instant',
            'qwen/qwen3-32b'
            ]
# also for groq
models_advanced = [
    'llama-3.3-70b-versatile',
    'openai/gpt-oss-120b'
]

In [3]:
# Data
VAL_DATA_PATH = "/content/val_BLvsLLM.csv"
RESULTS_PATH = "/content/llm_results.csv"

# 2. Init client and load data

In [4]:
# client = openai.OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=userdata.get('OPENROUTER_API_KEY'),
# )

client = openai.OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get('THESIS_API_KEY'),
)

In [5]:
val_df = pd.read_csv(VAL_DATA_PATH)
valid_classes_str = "\n".join([f"- {c}" for c in val_df['Misconception'].unique()])

In [7]:
valid_classes_str

'- Incomplete Calculation\n- Wrong Fraction\n- Unclassified Error\n- Duplication Error\n- Subtraction Error\n- Additive Reasoning Error\n- Unknowable Error\n- Swapped Dividend\n- Multiplication Error\n- Irrelevant Explanation\n- Positive/Negative Sign Error\n- Inversion Error\n- Whole Number Bias\n- Definition Error\n- Not A Variable\n- Denominator Only Change\n- Adding Fractions Across\n- Whole Numbers Are Larger\n- Wrong Term\n- Scale Factor Error\n- Ignores Zeroes\n- Multiplying By 4\n- Tacking On Zeroes\n- Shorter Decimals Are Bigger\n- First Term Error\n- Flip and Change Error\n- Interior Angle Error\n- Division Error\n- Base Rate Error\n- Adding Unlike Terms\n- Certainty Bias\n- Longer Decimals Are Bigger\n- Inverse Operation Error'

# 3 Prepare Session and Reload Results

In [ ]:
processed_indices = set()
if os.path.exists(RESULTS_PATH):
    results_df = pd.read_csv(RESULTS_PATH)
    processed_indices = set(results_df['index_id'].tolist())
    last_idx = max(processed_indices) if processed_indices else "None"
    print(f"Resuming session. Found {len(processed_indices)} completed rows.")
    print(f"Last processed index was: {last_idx}")
else:
    print("Starting fresh evaluation session.")

# Slice the DataFrame to only contain un-processed rows
remaining_df = val_df[~val_df.index.isin(processed_indices)]
print(f"Rows remaining to process: {len(remaining_df)} / {len(val_df)}")


Resuming session. Found 301 completed rows.
Last processed index was: 300
Rows remaining to process: 782 / 1083


In [ ]:
#results_df

# 4. Start the loop (openai)

In [ ]:
# Main Evaluation Loop
session_active = True
new_rows_this_session =[]

print("\nBeginning queries. Press 'Stop' in Colab if you want to manually pause.")

for index, row in remaining_df.iterrows():
    if not session_active:
        break

    problem_text = str(row.get('QuestionText', ''))
    student_ans = str(row.get('MC_Answer', ''))
    student_exp = str(row.get('StudentExplanation', ''))

    prompt = f"""You are a math tutor diagnostician.
Classify the student's error into EXACTLY ONE of these categories:
{valid_classes_str}
If vague, guessed, or no clear math error, output Unclassified_Error.

Problem: {problem_text}
Student Answer: {student_ans}
Explanation: {student_exp}

Respond in strict JSON format exactly like this: {{"prediction": "ClassName"}}"""

    # Dictionary to store the current row's data
    current_result = {
        'index_id': index,
        'True_Misconception': row['Misconception']
    }

    row_failed = False

    # Query all models for this single row
    for model_name in models_OR:
        try:
            # Sleep to respect Free Tier Limits
            time.sleep(10)

            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"}
            )

            # Parse output
            output_content = response.choices[0].message.content
            if output_content is None:
                    print(f"\n[-] {model_name} returned an empty/None response. Defaulting to Unclassified.")
                    pred = 'Unclassified_Error'
            else:
                    try:
                        pred = json.loads(output_content).get('prediction', 'Unclassified_Error')
                    except json.JSONDecodeError:
                        pred = 'Unclassified_Error' # Handled bad JSON strings

            current_result[model_name] = pred

        except openai.RateLimitError as e:
            # SPECIFIC 429 RATE LIMIT CATCHER
            print(f"\n[!] 🛑 RATE LIMIT (429) HIT ON MODEL: {model_name}")
            try:
                # Try to parse and pretty-print the exact OpenRouter JSON error
                error_body = e.response.json()
                print(json.dumps(error_body, indent=2))
            except:
                # Fallback if it's not JSON
                print(e.message)

            print("\n[!] Halting session. Check the error above to see if it's a minute or daily limit.")
            row_failed = True
            session_active = False
            break

        except openai.APIError as e:
            # SPECIFIC 500/502 SERVER ERROR CATCHER (e.g. Model is down)
            print(f"\n[!] ⚠️ UPSTREAM API ERROR ON MODEL: {model_name}")
            print(f"Status Code: {e.status_code}")
            print(f"Message: {e.message}")
            row_failed = True
            session_active = False
            break

        except Exception as e:
            # ANY OTHER ERROR
            print(f"\n[!] ❌ UNEXPECTED ERROR ON MODEL: {model_name}\n{str(e)}")
            row_failed = True
            session_active = False
            break

    # If the row completed all models successfully, save it to our session buffer
    if not row_failed:
        new_rows_this_session.append(current_result)
        print(f"Processed row index {index} successfully across all {len(models)} models.")

        # Save to disk periodically (every 5 rows) just in case Colab crashes/disconnects
        if len(new_rows_this_session) >= 5:
            temp_df = pd.DataFrame(new_rows_this_session)
            # Append if file exists, write with header if it doesn't
            if os.path.exists(RESULTS_PATH):
                temp_df.to_csv(RESULTS_PATH, mode='a', header=False, index=False)
            else:
                temp_df.to_csv(RESULTS_PATH, index=False)
            # Clear the buffer after saving
            new_rows_this_session =[]



Beginning queries. Press 'Stop' in Colab if you want to manually pause.

[!] 🛑 RATE LIMIT (429) HIT ON MODEL: meta-llama/llama-3.2-3b-instruct:free
{
  "error": {
    "message": "Provider returned error",
    "code": 429,
    "metadata": {
      "raw": "meta-llama/llama-3.2-3b-instruct:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations",
      "provider_name": "Venice",
      "is_byok": false
    }
  },
  "user_id": "user_2vKGKlVPhDhurJ5aGTASbS0YvtD"
}

[!] Halting session. Check the error above to see if it's a minute or daily limit.


In [ ]:
# 5. Final Save: When session halts (or finishes), save any remaining buffered rows
if len(new_rows_this_session) > 0:
    temp_df = pd.DataFrame(new_rows_this_session)
    if os.path.exists(RESULTS_PATH):
        temp_df.to_csv(RESULTS_PATH, mode='a', header=False, index=False)
    else:
        temp_df.to_csv(RESULTS_PATH, index=False)

print("\nSession Ended. All completed progress is safely saved in:", RESULTS_PATH)


Session Ended. All completed progress is safely saved in: /content/llm_results.csv


# 5. Prompt Preparation

DEFINITIONS OF AMBIGUOUS LABELS:
- Unclassified_Error: The text is vague (I guessed) or lacks clear math reasoning.
- Irrelevant: The text has absolutely nothing to do with the math problem.
- Incomplete: The student started doing the correct math but stopped halfway.
- WNB: (Whole Number Bias). Treating decimals/fractions as if they were whole numbers.
- SwapDividend: Reversing the order of division (e.g., calculating 6/2 instead of 2/6).

In [7]:
def create_few_shot_prompt(problem_text, student_ans, student_exp, valid_classes_str):
    template = r"""You are a meticulous educational analyst and expert mathematics tutor.
Your task is to diagnose the specific mathematical misconception in a student's incorrect answer.

DEFINITIONS OF SPECIFIC/AMBIGUOUS LABELS:
- Unclassified Error: The text is vague ("I worked it out in my head", "I guessed"), or the student gives no mathematical steps.
- Incomplete Calculation: The student did the correct math but stopped before finishing (e.g., failing to simplify a fraction).
- Additive Reasoning Error: Using addition/subtraction to solve proportional or multiplicative problems (e.g., solving equivalent fractions by adding instead of multiplying).
- Duplication Error: Multiplying BOTH the numerator and denominator of a fraction by a whole number.
- Swapped Dividend: Reversing the order of division to make it easier, usually dividing a larger number by a smaller one.
- Whole Number Bias: Treating parts of a fraction/decimal as independent whole numbers.
- Tacking On Zeroes: Performing math on absolute values and just "tacking on" a sign, symbol, or zero at the end without proper logical operations.

ALL VALID CLASSES (You MUST pick exactly one of these):
[VALID_CLASSES]

YOUR TASK:
1. Read the Problem, Student Answer, and Student Explanation.
2. Perform a THOUGHT ANALYSIS: Briefly explain what mathematical error the student made.
3. PREDICTION: Select the exact misconception class from the list above that matches your analysis.

=== EXAMPLES ===

Problem: \( \frac{X}{8} = \frac{7}{12} \) What is the value of \( X \)?
Student Answer: \( 3 \)
Explanation: 12 minus 4 is 8, so I did 7 minus 4 to get 3.
Output: {"thought": "The student incorrectly used additive reasoning (subtracting 4) instead of multiplicative reasoning to find the equivalent fraction.", "prediction": "Additive Reasoning Error"}

Problem: Calculate \( \frac{3}{4} \times 2 \)
Student Answer: \( \frac{6}{8} \)
Explanation: I multiplied the top by 2 and the bottom by 2.
Output: {"thought": "The student mistakenly multiplied both the numerator and the denominator by the whole number, creating an equivalent fraction rather than multiplying its value.", "prediction": "Duplication Error"}

Problem: Which number is the greatest? \( 4.5 \), \( 4.09 \), \( 4.12 \)
Student Answer: \( 4.5 \)
Explanation: I just looked at them and knew.
Output: {"thought": "The student provided the correct answer but gave a vague explanation with no mathematical reasoning to diagnose.", "prediction": "Unclassified Error"}

=== ACTUAL TASK ===

Problem: [PROBLEM_TEXT]
Student Answer: [STUDENT_ANS]
Explanation: [STUDENT_EXP]

CONSTRAINT:
Respond strictly in JSON format exactly like this: {"thought": "Your analysis here", "prediction": "ClassName"}
DO NOT wrap the JSON in markdown blocks (e.g., no ```json). Output RAW JSON only."""

    # Safely inject our variables using string replacement instead of f-strings
    prompt = template.replace("[VALID_CLASSES]", valid_classes_str)
    prompt = prompt.replace("[PROBLEM_TEXT]", problem_text)
    prompt = prompt.replace("[STUDENT_ANS]", student_ans)
    prompt = prompt.replace("[STUDENT_EXP]", student_exp)

    return prompt

In [8]:
test_obj = remaining_df.iloc[0]
test_prompt = create_few_shot_prompt(
        problem_text=test_obj['QuestionText'],
        student_ans=test_obj['MC_Answer'],
        student_exp=test_obj['StudentExplanation'],
        valid_classes_str=valid_classes_str
    )
print(test_prompt)

You are a meticulous educational analyst and expert mathematics tutor.
Your task is to diagnose the specific mathematical misconception in a student's incorrect answer.

DEFINITIONS OF SPECIFIC/AMBIGUOUS LABELS:
- Unclassified Error: The text is vague ("I worked it out in my head", "I guessed"), or the student gives no mathematical steps.
- Incomplete Calculation: The student did the correct math but stopped before finishing (e.g., failing to simplify a fraction).
- Additive Reasoning Error: Using addition/subtraction to solve proportional or multiplicative problems (e.g., solving equivalent fractions by adding instead of multiplying).
- Duplication Error: Multiplying BOTH the numerator and denominator of a fraction by a whole number.
- Swapped Dividend: Reversing the order of division to make it easier, usually dividing a larger number by a smaller one.
- Whole Number Bias: Treating parts of a fraction/decimal as independent whole numbers.
- Tacking On Zeroes: Performing math on absol

# 6. Start the loop (Groq)

In [9]:
# Main Evaluation Loop
session_active = True
new_rows_this_session =[]

print("\nBeginning Groq queries. Press 'Stop' in Colab to manually pause.")

for index, row in remaining_df.iterrows():
    if not session_active:
        break

    problem_text = str(row.get('QuestionText', ''))
    student_ans = str(row.get('MC_Answer', ''))
    student_exp = str(row.get('StudentExplanation', ''))

    # NOTE: Groq REQUIRES the word "JSON" in the prompt to use JSON mode.
    prompt = create_few_shot_prompt(
        problem_text=problem_text,
        student_ans=student_ans,
        student_exp=student_exp,
        valid_classes_str=valid_classes_str
    )

    current_result = {
        'index_id': index,
        'True_Misconception': row['Misconception']
    }

    row_failed = False

    for model_name in models_advanced: #new model set
        try:
            # Groq Free Tier allows ~30 requests per minute.
            # 3 seconds per request guarantees we stay safely under the limit!
            time.sleep(3)

            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"} # Groq loves this format
            )

            output_content = response.choices[0].message.content

            if output_content is None:
                print(f"\n[-] {model_name} returned None. Defaulting to Unclassified.")
                pred = 'Unclassified Error'
            else:
                try:
                    pred = json.loads(output_content).get('prediction', 'Unclassified Error')
                except json.JSONDecodeError:
                    pred = 'Unclassified Error'

            current_result[model_name] = pred

        except openai.RateLimitError as e:
            # GROQ MINUTE/DAILY LIMIT HIT
            print(f"\n[!] 🛑 RATE LIMIT HIT ON: {model_name}")
            print("[!] Halting session. Wait a minute and run again.")
            row_failed = True
            session_active = False
            break

        except openai.BadRequestError as e:
            # CATCH GROQ 400 JSON ERRORS HERE (NO RETRIES)
            if "json_validate_failed" in str(e):
                print(f"\n[-] {model_name} failed Groq JSON validation. Defaulting to Unclassified.")
                current_result[model_name] = 'Unclassified Error'
                # Gracefully accept defeat. Does NOT halt the session. Moves to next model.
            else:
                print(f"\n[!] ❌ BAD REQUEST ON MODEL: {model_name}\n{str(e)}")
                row_failed = True
                session_active = False
                break

        except Exception as e:
            print(f"\n[!] ❌ ERROR ON MODEL: {model_name}\n{str(e)}")
            row_failed = True
            session_active = False
            break

    if not session_active:
        break # Break main dataset loop if session halted

    # If the row completed all models successfully, save it to our session buffer
    if not row_failed:
        new_rows_this_session.append(current_result)
        # Fixed print statement to use models_G
        print(f"Processed row index {index} successfully across all {len(models_G)} models.")

        # Save to disk periodically (every 5 rows) just in case Colab crashes/disconnects
        if len(new_rows_this_session) >= 5:
            temp_df = pd.DataFrame(new_rows_this_session)
            # Append if file exists, write with header if it doesn't
            if os.path.exists(RESULTS_PATH):
                temp_df.to_csv(RESULTS_PATH, mode='a', header=False, index=False)
            else:
                temp_df.to_csv(RESULTS_PATH, index=False)
            # Clear the buffer after saving
            new_rows_this_session =[]

# Final Save: When session halts (or finishes), save any remaining buffered rows
if len(new_rows_this_session) > 0:
    temp_df = pd.DataFrame(new_rows_this_session)
    if os.path.exists(RESULTS_PATH):
        temp_df.to_csv(RESULTS_PATH, mode='a', header=False, index=False)
    else:
        temp_df.to_csv(RESULTS_PATH, index=False)
    new_rows_this_session = []


Beginning Groq queries. Press 'Stop' in Colab to manually pause.
Processed row index 301 successfully across all 3 models.
Processed row index 302 successfully across all 3 models.
Processed row index 303 successfully across all 3 models.
Processed row index 304 successfully across all 3 models.
Processed row index 305 successfully across all 3 models.
Processed row index 306 successfully across all 3 models.
Processed row index 307 successfully across all 3 models.
Processed row index 308 successfully across all 3 models.
Processed row index 309 successfully across all 3 models.
Processed row index 310 successfully across all 3 models.
Processed row index 311 successfully across all 3 models.
Processed row index 312 successfully across all 3 models.
Processed row index 313 successfully across all 3 models.
Processed row index 314 successfully across all 3 models.
Processed row index 315 successfully across all 3 models.
Processed row index 316 successfully across all 3 models.
Proces

# 7. Metric evaluation

In [10]:
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display

In [ ]:
if not os.path.exists(RESULTS_PATH):
    print("No results file found yet. Wait for the loop to save the first batch!")
else:
    # Load the intermediate results
    results_df = pd.read_csv(RESULTS_PATH)
    print(f"--- Intermediate Results: {len(results_df)} rows evaluated so far ---\n")

    # Extract ground truth
    y_true = results_df['True_Misconception'].astype(str)

    # Identify the model columns
    ignore_cols = ['index_id', 'True_Misconception']
    model_columns =[col for col in results_df.columns if col not in ignore_cols]

    # Calculate metrics for each model
    metrics_list =[]

    for model_name in model_columns:
        y_pred = results_df[model_name].astype(str)

        acc = accuracy_score(y_true, y_pred)
        f1_mac = f1_score(y_true, y_pred, average='macro')
        f1_wei = f1_score(y_true, y_pred, average='weighted')

        metrics_list.append({
            'Model': model_name,
            'F1-Macro': f1_mac,
            'F1-Weighted': f1_wei,
            'Accuracy': acc,
        })

    # Display metrics
    metrics_df = pd.DataFrame(metrics_list)
    metrics_df = metrics_df.sort_values(by='F1-Macro', ascending=False).reset_index(drop=True)

    # Apply styling
    styled_df = metrics_df.style.background_gradient(cmap='Blues', subset=[ 'F1-Macro', 'F1-Weighted','Accuracy']) \
                                .format({'F1-Macro': '{:.4f}', 'F1-Weighted': '{:.4f}','Accuracy': '{:.4f}'})

    display(styled_df)

--- Intermediate Results: 401 rows evaluated so far ---



,Model,F1-Macro,F1-Weighted,Accuracy
0,openai/gpt-oss-120b,0.2399,0.5180,0.4913
1,llama-3.3-70b-versatile,0.1460,0.4230,0.3940
